# 🔭 GalaxyMorph — Hybrid CNN+Transformer for Galaxy Morphology

**Arquitectura:** EfficientNet-B0 (backbone) + Transformer Encoder (2 capas, 8 heads)

**Acelerador:** GPU T4 x2 (DataParallel)

| Idx | Clase | Descripcion |
|-----|-------|-------------|
| 0 | Elliptical | Galaxias elípticas sin estructura |
| 1 | Spiral | Galaxias espirales con brazos visibles |
| 2 | Barred_Spiral | Espirales con barra central |
| 3 | Edge_on | Galaxias vistas de perfil |
| 4 | Irregular_Merger | Galaxias irregulares o en fusión |

## 1. Setup & Imports

In [ ]:
import os, gc, json, math, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torchvision
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.transforms import v2 as transforms

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB)")

## 2. Configuración

In [ ]:
# Rutas Kaggle
KAGGLE_IMAGES_DATASET = "/kaggle/input/datasets/jaimetrickz/galaxy-zoo-2-images/images_gz2"
KAGGLE_CSV_DATASET = "/kaggle/input/datasets/jeancdevx/mis-archivos"
DATASET_CSV = os.path.join(KAGGLE_CSV_DATASET, "galaxy_dataset.csv")
LOCAL_IMAGES_DIR = os.path.join(KAGGLE_IMAGES_DATASET, "images")
CHECKPOINT_DIR = "/kaggle/working/models/checkpoints"
LOG_DIR = "/kaggle/working/logs"

# Hiperparámetros
BATCH_SIZE = 128
EPOCHS = 40
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-3
WARMUP_EPOCHS = 3
PHASE1_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 7
LABEL_SMOOTHING = 0.1
MIXUP_ALPHA = 0.2
NUM_CLASSES = 5
INPUT_SIZE = 224

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | Batch: {BATCH_SIZE}")

## 3. Exploración del Dataset

In [ ]:
df = pd.read_csv(DATASET_CSV)
class_names = ["Elliptical", "Spiral", "Barred_Spiral", "Edge_on", "Irregular_Merger"]
class_colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]
print(f"Total: {len(df):,} muestras")
for label in class_names:
    c = (df['label'] == label).sum()
    print(f"  {label:20s}: {c:>7,} ({c/len(df)*100:.1f}%)")
print(f"\nSplits: {df['split'].value_counts().to_dict()}")

### Distribución de Clases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
class_counts = df["label"].value_counts()
bars = axes[0].bar(range(len(class_counts)), class_counts.values, color=class_colors)
axes[0].set_xticks(range(len(class_counts)))
axes[0].set_xticklabels(class_counts.index, rotation=25, ha="right")
axes[0].set_title("Distribución de Clases")
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2., bar.get_height()+500, f"{count:,}", ha="center", fontsize=9, fontweight="bold")
axes[1].pie(class_counts.values, labels=class_counts.index, colors=class_colors, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Proporción")
plt.tight_layout(); plt.show()

### 🔭 Galería de Galaxias

In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(14, 18))
fig.suptitle("Galería de Galaxias por Clase", fontsize=16, fontweight="bold")
for row, (label, color) in enumerate(zip(class_names, class_colors)):
    subset = df[df["label"]==label].sample(n=4, random_state=42)
    for col, (_, s) in enumerate(subset.iterrows()):
        ax = axes[row, col]
        try:
            ax.imshow(Image.open(Path(LOCAL_IMAGES_DIR)/Path(s["image_path"]).name))
        except: ax.text(0.5, 0.5, "N/A", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"{label}\n{s['gz2_class']}", fontsize=8, color=color); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Modelo
### 4.1 Utilidades

In [ ]:
class MetricsTracker:
    def __init__(self, num_classes=5, class_names=None):
        self.num_classes = num_classes
        self.class_names = class_names or [f"C{i}" for i in range(num_classes)]
        self.reset()
    def reset(self):
        self.preds, self.tgts = [], []
    def update(self, logits, targets):
        self.preds.extend(logits.argmax(1).cpu().numpy())
        self.tgts.extend(targets.cpu().numpy())
    def compute(self):
        if not self.preds: return {}
        p, t = np.array(self.preds), np.array(self.tgts)
        return {"accuracy": accuracy_score(t,p), "f1_macro": f1_score(t,p,average="macro",zero_division=0), "f1_weighted": f1_score(t,p,average="weighted",zero_division=0)}
    def confusion_matrix(self):
        return confusion_matrix(np.array(self.tgts), np.array(self.preds), labels=list(range(self.num_classes))) if self.preds else np.zeros((self.num_classes,self.num_classes))

def get_class_weights(csv_path):
    m = pd.read_csv(csv_path)
    m = m[m["split"]=="train"]
    cc = m["label_idx"].value_counts().sort_index()
    w = len(m) / (len(cc) * cc.values)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32)

def get_sample_weights(csv_path, split="train"):
    m = pd.read_csv(csv_path)
    m = m[m["split"]==split].reset_index(drop=True)
    cc = m["label_idx"].value_counts()
    cw = len(m) / (len(cc) * cc)
    sw = m["label_idx"].map(cw).values
    return sw / sw.sum() * len(sw)

print("✅ Utilidades definidas")

### 4.2 Dataset & Transforms

In [ ]:
class GalaxyMorphDataset(Dataset):
    def __init__(self, csv_path, split="train", transform=None):
        self.transform = transform
        self.manifest = pd.read_csv(csv_path)
        self.manifest = self.manifest[self.manifest["split"]==split].reset_index(drop=True)
        print(f"  {split}: {len(self.manifest):,} samples")
    def __len__(self): return len(self.manifest)
    def __getitem__(self, idx):
        while True:
            row = self.manifest.iloc[idx]
            try:
                img = Image.open(Path(LOCAL_IMAGES_DIR)/Path(str(row["image_path"])).name).convert("RGB")
                break
            except: idx = random.randint(0, len(self.manifest)-1)
        if self.transform: img = self.transform(img)
        return img, int(row["label_idx"])

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=180),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
val_transforms = transforms.Compose([
    transforms.Resize(INPUT_SIZE + 32),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
print("✅ Dataset & transforms definidos")

### 4.3 Modelo Híbrido: EfficientNet-B0 + Transformer Encoder

In [ ]:
class GalaxyMorphHybrid(nn.Module):
    def __init__(self, num_classes=5, pretrained=True, t_dim=512, t_heads=8, t_layers=2, t_drop=0.3, h_drop=0.5):
        super().__init__()
        self.backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        self.backbone.classifier = nn.Identity()
        self.backbone.avgpool = nn.Identity()

        self.projection = nn.Linear(1280, t_dim)
        self.pos_embed = nn.Parameter(torch.randn(1, 49, t_dim) * 0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=t_dim, nhead=t_heads, dim_feedforward=t_dim*4,
            dropout=t_drop, activation="gelu", batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=t_layers)

        self.head = nn.Sequential(nn.LayerNorm(t_dim), nn.Dropout(h_drop), nn.Linear(t_dim, num_classes))
        self._init()

    def _init(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.xavier_uniform_(self.projection.weight); nn.init.zeros_(self.projection.bias)
        for m in self.head.modules():
            if isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        f = self.backbone.features(x)              # (B,1280,7,7)
        t = f.flatten(2).transpose(1,2)             # (B,49,1280)
        t = self.projection(t) + self.pos_embed     # (B,49,512)
        t = self.transformer(t)                     # (B,49,512)
        return self.head(t.mean(dim=1))             # (B,5)

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
        print("🧊 Backbone congelado")

    def unfreeze_backbone(self, n=3):
        total = len(self.backbone.features)
        for i, blk in enumerate(self.backbone.features):
            if i >= total - n:
                for p in blk.parameters(): p.requires_grad = True
        ct = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        print(f"🔥 Últimos {n} bloques descongelados ({ct:,} params)")

    def get_param_groups(self, bb_lr=1e-5, head_lr=1e-4):
        bb, other = [], []
        for name, p in self.named_parameters():
            if not p.requires_grad: continue
            (bb if name.startswith("backbone") else other).append(p)
        return [{"params": bb, "lr": bb_lr}, {"params": other, "lr": head_lr}]

print("✅ Modelo definido")

### 4.4 Resumen

In [ ]:
_m = GalaxyMorphHybrid().to(device)
total = sum(p.numel() for p in _m.parameters())
_m.freeze_backbone()
p1 = sum(p.numel() for p in _m.parameters() if p.requires_grad)
_m.unfreeze_backbone(3)
p2 = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"Total params:     {total:,}")
print(f"Phase 1 trainable: {p1:,}")
print(f"Phase 2 trainable: {p2:,}")
with torch.no_grad():
    o = _m(torch.randn(2,3,224,224).to(device))
    print(f"Forward: (2,3,224,224) → {o.shape} ✅")
del _m, o; torch.cuda.empty_cache()

## 5. Entrenamiento
### 5.1 DataLoaders

In [ ]:
print("Cargando datasets...")
train_ds = GalaxyMorphDataset(DATASET_CSV, "train", train_transforms)
val_ds = GalaxyMorphDataset(DATASET_CSV, "val", val_transforms)

sw = get_sample_weights(DATASET_CSV, "train")
sampler = WeightedRandomSampler(sw, len(sw), replacement=True, generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print(f"Train: {len(train_loader)} batches | Val: {len(val_loader)} batches")

### 5.2 Funciones de Entrenamiento

In [ ]:
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1-lam) * x[idx], y, y[idx], lam

def mixup_criterion(crit, pred, ya, yb, lam):
    return lam * crit(pred, ya) + (1-lam) * crit(pred, yb)

def lr_schedule(step, warmup, total):
    if step < warmup: return (step + 1) / warmup
    prog = (step - warmup) / max(1, total - warmup)
    return max(0.01, 0.5 * (1 + math.cos(math.pi * prog)))

def train_one_epoch(model, loader, criterion, optimizer, device, use_mixup=False, alpha=0.2):
    model.train()
    total_loss = 0.0
    metrics = MetricsTracker(NUM_CLASSES, class_names)
    pbar = tqdm(loader, desc="  Train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        if use_mixup:
            images, ya, yb, lam = mixup_data(images, labels, alpha)
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, ya, yb, lam)
            metrics.update(outputs.detach(), ya)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            metrics.update(outputs.detach(), labels)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / len(loader), metrics.compute()

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    metrics = MetricsTracker(NUM_CLASSES, class_names)
    for images, labels in tqdm(loader, desc="  Val  ", leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        metrics.update(outputs, labels)
    return total_loss / len(loader), metrics.compute()

print("✅ Funciones definidas")

### 5.3 Setup

In [ ]:
model = GalaxyMorphHybrid(num_classes=NUM_CLASSES)
if torch.cuda.device_count() > 1:
    print(f"🖥️  {torch.cuda.device_count()} GPUs → DataParallel")
    model = nn.DataParallel(model)
model = model.to(device)
base_model = model.module if isinstance(model, nn.DataParallel) else model

class_weights = get_class_weights(DATASET_CSV).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)

history = {"train_loss":[],"train_accuracy":[],"train_f1_macro":[],"val_loss":[],"val_accuracy":[],"val_f1_macro":[],"val_f1_weighted":[],"lr":[]}
print(f"Class weights: {[f'{w:.3f}' for w in class_weights.tolist()]}")
print("✅ Setup completo")

### 5.4 Entrenar 🚀

**Fase 1** (epochs 1-10): Backbone congelado

**Fase 2** (epochs 11-40): Últimos 3 bloques descongelados + MixUp

In [ ]:
best_f1, best_ep, patience = 0.0, 0, 0
optimizer = None
p1_step, p2_step = 0, 0

for epoch in range(1, EPOCHS + 1):
    torch.cuda.empty_cache(); gc.collect()

    # ── Phase transitions ──
    if epoch == 1:
        base_model.freeze_backbone()
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"\n{'='*70}")
        print(f"  FASE 1: Backbone CONGELADO (epochs 1-{PHASE1_EPOCHS}) | {tr:,} params")
        print(f"{'='*70}")

    elif epoch == PHASE1_EPOCHS + 1:
        base_model.unfreeze_backbone(3)
        optimizer = optim.AdamW(
            base_model.get_param_groups(bb_lr=1e-5, head_lr=LEARNING_RATE),
            weight_decay=WEIGHT_DECAY)
        patience = 0
        tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"\n{'='*70}")
        print(f"  FASE 2: DESCONGELADO PARCIAL (epochs {PHASE1_EPOCHS+1}-{EPOCHS}) | {tr:,} params")
        print(f"{'='*70}")

    # ── LR schedule (always from BASE lr) ──
    if epoch <= PHASE1_EPOCHS:
        mult = lr_schedule(p1_step, WARMUP_EPOCHS, PHASE1_EPOCHS)
        for pg in optimizer.param_groups: pg['lr'] = LEARNING_RATE * mult
        p1_step += 1
    else:
        mult = lr_schedule(p2_step, 0, EPOCHS - PHASE1_EPOCHS)
        for i, pg in enumerate(optimizer.param_groups):
            pg['lr'] = (1e-5 if i == 0 else LEARNING_RATE) * mult
        p2_step += 1

    cur_lr = optimizer.param_groups[-1]['lr']

    # ── Train & Validate ──
    t0 = time.time()
    use_mixup = epoch > PHASE1_EPOCHS
    train_loss, train_m = train_one_epoch(model, train_loader, criterion, optimizer, device, use_mixup, MIXUP_ALPHA)
    val_loss, val_m = validate(model, val_loader, criterion, device)
    elapsed = time.time() - t0

    # ── Record ──
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_m.get("accuracy",0))
    history["train_f1_macro"].append(train_m.get("f1_macro",0))
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_m.get("accuracy",0))
    history["val_f1_macro"].append(val_m.get("f1_macro",0))
    history["val_f1_weighted"].append(val_m.get("f1_weighted",0))
    history["lr"].append(cur_lr)

    vf1 = val_m.get("f1_macro",0)
    ph = "P1" if epoch <= PHASE1_EPOCHS else "P2"
    line = (f"Epoch {epoch:>2}/{EPOCHS} [{ph}] | LR:{cur_lr:.2e} | "
            f"Train[loss={train_loss:.4f} f1={train_m.get('f1_macro',0):.4f}] | "
            f"Val[loss={val_loss:.4f} f1={vf1:.4f} acc={val_m.get('accuracy',0):.4f}] | {elapsed:.0f}s")

    if vf1 > best_f1:
        best_f1, best_ep, patience = vf1, epoch, 0
        torch.save({"model_state_dict": base_model.state_dict(), "epoch": epoch,
                     "best_val_f1_macro": best_f1, "history": history},
                    os.path.join(CHECKPOINT_DIR, "best_model.pt"))
        line += " ✅ BEST"
    else: patience += 1
    print(line)

    if epoch > PHASE1_EPOCHS and patience >= EARLY_STOPPING_PATIENCE:
        print(f"\n⛔ Early stopping epoch {epoch}")
        break

print(f"\n{'='*70}")
print(f"  Mejor Val F1: {best_f1:.4f} (epoch {best_ep})")
print(f"{'='*70}")
with open(os.path.join(LOG_DIR, "training_history.json"), "w") as f:
    json.dump(history, f, indent=2)
print("Historial guardado")

## 6. Resultados

In [ ]:
er = range(1, len(history["train_loss"])+1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, tk, vk, title in [
    (axes[0], "train_loss", "val_loss", "Loss"),
    (axes[1], "train_accuracy", "val_accuracy", "Accuracy"),
    (axes[2], "train_f1_macro", "val_f1_macro", "F1 (macro)")]:
    ax.plot(er, history[tk], label="Train", color="#e74c3c", lw=2)
    ax.plot(er, history[vk], label="Val", color="#3498db", lw=2)
    ax.axvline(x=PHASE1_EPOCHS, color="gray", ls="--", alpha=0.7, label="P1→P2")
    if title == "F1 (macro)":
        ax.axhline(y=0.6939, color="orange", ls=":", alpha=0.8, label="ResNet50 baseline")
    ax.set_xlabel("Epoch"); ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle("GalaxyMorphHybrid Results", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()
print(f"Best Val F1: {best_f1:.4f} (epoch {best_ep})")
if history['train_f1_macro']:
    print(f"Gap: {history['train_f1_macro'][-1] - history['val_f1_macro'][-1]:.4f}")

### 6.1 Confusion Matrix

In [ ]:
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, "best_model.pt"), map_location=device, weights_only=False)
em = GalaxyMorphHybrid(num_classes=NUM_CLASSES).to(device); em.load_state_dict(ckpt["model_state_dict"]); em.eval()
mt = MetricsTracker(NUM_CLASSES, class_names)
with torch.no_grad():
    for imgs, lbls in val_loader:
        mt.update(em(imgs.to(device)), lbls.to(device))
cm = mt.confusion_matrix()
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix — Best Epoch {ckpt['epoch']}")
plt.tight_layout(); plt.show()
f = mt.compute()
print(f"Acc: {f['accuracy']:.4f} | F1 macro: {f['f1_macro']:.4f} | F1 weighted: {f['f1_weighted']:.4f}")
del em; torch.cuda.empty_cache()

## 7. Descargar Modelo

In [ ]:
from IPython.display import FileLink
p = os.path.join(CHECKPOINT_DIR, "best_model.pt")
if os.path.exists(p):
    print(f"📦 {p} ({os.path.getsize(p)/1e6:.1f} MB)")
    FileLink(p)
else: print("⚠️ No encontrado")